In [6]:
import pandas as pd

#loading the csv
df = pd.read_csv("montreal_travel_time.csv")

#convert the timestamp text into a smart Pandas datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'])

#Extract the 'hour' to match with weather data
df['hour'] = df['timestamp'].dt.floor('h')

print(f"Loaded {len(df)} transit segments!")
df.head()

Loaded 5914 transit segments!


,timestamp,route_id,travel_seconds,travel_minutes,hour
0,2026-06-10 19:33:40,24,57,0.95,2026-06-10 19:00:00
1,2026-06-10 19:33:40,166,108,1.80,2026-06-10 19:00:00
2,2026-06-10 19:33:40,71,50,0.83,2026-06-10 19:00:00
3,2026-06-10 19:33:40,18,58,0.97,2026-06-10 19:00:00
4,2026-06-10 19:33:40,747,52,0.87,2026-06-10 19:00:00


In [7]:
import meteostat as ms
from datetime import datetime, timedelta

end_date = datetime.now() 
start_date = end_date - timedelta(days=10)

weather_df = ms.hourly(
    station='71627',
    start=start_date,
    end=end_date
).fetch().reset_index()
weather_df = weather_df.fillna(0)

print(weather_df.shape)
weather_df.head()

(240, 12)


,time,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
0,2026-05-31 21:00:00,14.5,62,0.0,0,70,26.0,20.0,0.0,0,2,3
1,2026-05-31 22:00:00,15.1,63,0.0,0,190,14.0,31.5,0.0,0,6,4
2,2026-05-31 23:00:00,15.6,58,0.0,0,20,14.0,31.5,0.0,0,6,3
3,2026-06-01 00:00:00,14.1,63,0.0,0,40,15.0,31.5,0.0,0,4,2
4,2026-06-01 01:00:00,12.9,70,0.0,0,20,13.0,31.5,0.0,0,2,2


In [8]:
import pandas as pd

transit_df = pd.read_csv('montreal_travel_time.csv')

transit_df['timestamp'] = pd.to_datetime(transit_df['timestamp'])
weather_df['time'] = pd.to_datetime(weather_df['time'])

transit_df['merge_hour'] = transit_df['timestamp'].dt.floor('h')

#merging
final_dataset = pd.merge(
    left=transit_df,
    right=weather_df,
    left_on='merge_hour',
    right_on='time',
    how='left'
)

final_dataset = final_dataset.drop(columns=['merge_hour','time'])

print(final_dataset.shape)
final_dataset.head()

(5914, 15)


,timestamp,route_id,travel_seconds,travel_minutes,temp,rhum,prcp,snwd,wdir,wspd,wpgt,pres,tsun,cldc,coco
0,2026-06-10 19:33:40,24,57,0.95,23.0,84,0.8,0,111,11.1,33.3,1007.7,0,8,7
1,2026-06-10 19:33:40,166,108,1.80,23.0,84,0.8,0,111,11.1,33.3,1007.7,0,8,7
2,2026-06-10 19:33:40,71,50,0.83,23.0,84,0.8,0,111,11.1,33.3,1007.7,0,8,7
3,2026-06-10 19:33:40,18,58,0.97,23.0,84,0.8,0,111,11.1,33.3,1007.7,0,8,7
4,2026-06-10 19:33:40,747,52,0.87,23.0,84,0.8,0,111,11.1,33.3,1007.7,0,8,7


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

#extract raw date and time numbers from the timestamp
final_dataset['Hour'] = final_dataset['timestamp'].dt.hour
final_dataset['DayOfWeek'] = final_dataset['timestamp'].dt.dayofweek # 0=Monday, 6=Sunday

#define codomain and domain for the model
y = final_dataset['travel_minutes']
X = final_dataset.drop(columns=['timestamp', 'route_id', 'travel_seconds', 'travel_minutes'])

#rewrite every column name as pure text
X.columns = [str(c) for c in X.columns]

#split the data into 80% train and 20% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#train the model with the train data
model = LinearRegression()
print("Training the model...")
model.fit(X_train, y_train)
print("Training complete! The AI has learned the traffic patterns.")

#test the model against the test data
predictions = model.predict(X_test)

#analyze the results
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("-" * 30)
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")

Training the model...
Training complete! The AI has learned the traffic patterns.
------------------------------
Mean Absolute Error (MAE): 1.29 minutes
Root Mean Squared Error (RMSE): 2.87 minutes


In [12]:
import csv
import os

from datetime import datetime
# This creates a timestamp specific to the moment you run the notebook
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Define the log file name
log_file = 'performance_log.csv'

rounded_mae = round(mae, 2)
rounded_rmse = round(rmse, 2)


# Prepare the data to write
file_exists = os.path.isfile(log_file)
data = [timestamp, len(final_dataset), rounded_mae, rounded_rmse]

# Append to the log
with open(log_file, 'a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['timestamp', 'num_rows', 'mae', 'rmse'])
    writer.writerow(data)

print(f"Performance logged to {log_file}")

Performance logged to performance_log.csv
